# Dataset Regeneration v2

Genera `data/dataset_regenerado_v2_10000_clientes.csv` con las políticas v2 (abajo). La comparación contra el dataset original está en `EDA_regenerado_v2.ipynb`.

> ⚠️ El código de este notebook fue generado por una **IA (Claude, de Anthropic)** a partir de las políticas escritas por el equipo.

## Políticas v2

### Customer ID (`customer_id`)
Formato `C######` (por ejemplo `C000001`), asignados en orden. Sin cambios.

### Edad (`edad`) — raíz
Se conserva el rango del dataset original, de 18 a 75 años: no hay menores de edad con tarjeta de crédito. Dentro de ese rango, cada grupo pesa lo que pesa en la pirámide poblacional de México 2026:

| Grupo | Población | Nota |
|---|---|---|
| 18-24 | 15,571,098 | 7/10 del grupo 15-24 |
| 25-34 | 21,105,726 | |
| 35-44 | 18,729,814 | |
| 45-54 | 15,640,829 | |
| 55-64 | 12,087,127 | |
| 65-74 | 7,384,543 | |
| 75 | 270,410 | 1/16 del grupo 75+ |

La edad exacta se muestrea uniforme dentro de cada grupo. Fuente: [PopulationPyramid México 2026](https://www.populationpyramid.net/es/m%C3%A9xico/2026/)

### Satisfacción (`satisfaccion`) — raíz
Normal con media 4 y desviación 0.6, en [1, 5] con un decimal. La media sale del 80.1 % de satisfacción del informe anual de Santander 2025. La desviación sube de 0.5 a 0.6 para que haya clientes insatisfechos.

### Antigüedad en meses (`antiguedad_meses`)
Nadie puede ser cliente antes de los 18 años, y se conserva el máximo de 15 años del original.

`uniforme(1, min(180, (edad − 18)·12 + 1))`

### Ingreso mensual (`ingreso_mensual`)
Log-normal con la mediana del original (27,140 MXN) y σ_log = 0.47 → media ≈ 30,600, p5 ≈ 11,700 y p95 ≈ 61,500, como el original. El ingreso sube hasta los 45-54 años y luego baja con las jubilaciones:

| 18-24 | 25-34 | 35-44 | 45-54 | 55-64 | 65-74 | 75 |
|---|---|---|---|---|---|---|
| 0.62 | 0.88 | 1.06 | 1.15 | 1.08 | 0.86 | 0.80 |

`29,100 · multiplicador(edad) · exp(N(0, 0.47))`

### Gasto mensual (`gasto_mensual`)
En promedio se gasta el 39 % del ingreso, como en el original (12,050 / 30,627). Techo de 110 % del ingreso.

`min(0.39 · ingreso · exp(N(0, 0.25) − 0.25²/2), 1.1 · ingreso)`

### NPS (`nps`)
Depende de la satisfacción. Media 6.92 (informe Santander 2025), entero en [0, 10].

`clamp(0, round(6.92 + 2.6·(satisfacción − 4) + N(0, 1.4)), 10)`

### Totalero (variable interna, no se guarda)
Banxico separa a quien paga todo cada mes (totalero) de quien arrastra deuda. En Santander, el 30.8 % de las tarjetas son no-totaleras (Cuadro 7). La probabilidad de ser totalero baja si el gasto pesa mucho sobre el ingreso y sube con el ingreso y la edad. Mueve pagos puntuales, uso de la línea y promociones.

`P(totalero) = logística(0.80 − 4·(gasto/ingreso − 0.39) + 0.6·ln(ingreso/27,140) + 0.012·(edad − 40))` → ≈ 68.4 % de totaleros.

### Transacciones al mes (`transacciones_mes`)
Proporcionales al gasto, con el ticket promedio del original (≈ 670 MXN). Poisson porque es un conteo.

`máx(1, Poisson(18 · (gasto / 11,100)^0.3))`

### Uso e-commerce (`ecommerce_pct`)
Sube con el gasto y baja con la edad; media de 50 % como el original. Rango [0, 100].

`clamp(0, 50 + 12·z(ln gasto) − 0.55·(edad − 41) + N(0, 20), 100)`

### Logins al mes (`app_logins_mes`)
Proporcionales al e-commerce y a las transacciones. Poisson con media 10, como el original.

`máx(1, Poisson(2.5 + 0.06·ecommerce + 0.24·transacciones))`

### Pagos puntuales (`pagos_puntuales_pct`)
En [70, 100] como el original. Los totaleros casi nunca se atrasan; los no-totaleros más, sobre todo si su gasto pesa mucho sobre su ingreso. Calibración Banxico: 4.9 % de tarjetas atrasadas (Cuadro 11) y 4.5 % de no-totaleros sin cubrir el mínimo → ≈ 5.4 % de clientes con menos de 80 % de pagos a tiempo.

- Totalero: `100 − Exponencial(1.5)`
- No-totalero: `100 − Exponencial(11) − 12·máx(0, gasto/ingreso − 0.39)`
- Ambos recortados a [70, 100].

### Uso de tarjeta (`uso_tarjeta_pct`)
Parte usada de la línea (saldo / límite). En Santander el saldo promedio es 27,300 MXN contra un límite de ≈ 76,600 MXN, ≈ 36 % (Cuadros 4, 8, 9 y 10). Los no-totaleros usan más su línea. Sube con las transacciones. Estrictamente en (0, 100).

*Cambio respecto a la política anterior:* ya no sube con el ingreso. Según Banxico, las tarjetas de mayor nivel tienen límites más altos en proporción a su saldo: el uso es de 33 % en Clásicas y de 27 % en Platino.

`Beta(media = (0.24 si totalero, 0.60 si no) · (transacciones / 18)^0.3, concentración 6)`, en porcentaje.

### Abandono (`abandono_riesgo`)
Inversamente proporcional a la antigüedad, el NPS y la satisfacción. Se agregan dos señales: usar menos del 10 % de la línea (las cuentas inactivas son el 17.5 % según Banxico, Cuadro 11) y los atrasos en pagos. Escala 0 a 1, media ≈ 0.175.

- `lealtad = 40·min(antigüedad, 36)/36 + 30·nps/10 + 30·satisfacción/5`
- `riesgo = logística(1.12 − 0.045·lealtad + 1.3·[uso < 10] + 0.06·(100 − pagos) + N(0, 0.6))`

### Segmento (`segmento`)
Basado en gasto y antigüedad. Umbrales calibrados para que la mezcla de niveles de tarjeta coincida con Banxico (Clásica 59.0 %, Oro 31.1 %, Platino 9.8 %, Cuadro 1) y para que Mass siga dominando como en el original.

| Segmento | Regla | % esperado |
|---|---|---|
| Premier | gasto ≥ 22,000 y antigüedad ≥ 24 m | 7.0 % |
| Select | gasto ≥ 15,500 y antigüedad ≥ 12 m | 14.2 % |
| Mass | el resto | 78.8 % |

### Tarjeta Santander (`tarjeta_santander`)
Mismo árbol de la v1, con umbrales nuevos. Dentro de cada nivel se conservan las proporciones entre tarjetas del original.

- **Mass**
  - Ingreso ≥ 31,000 (el 25 % de mayor ingreso del segmento): Aeroméxico Blanca o Fiesta Rewards Oro (63 % / 37 %).
  - Pocos ingresos y gasto ≥ 7,000: LikeU.
  - Pocos ingresos y gasto < 7,000: Free.
- **Select**
  - Gasto ≥ media del segmento y antigüedad ≥ 72 m: Santander / Fiesta Rewards / Aeroméxico Platinum (43 / 22 / 35 %).
  - El resto: Santander Gold.
- **Premier**
  - Ingreso ≥ 60,000, antigüedad ≥ 60 m y gasto ≥ 30,000: World Elite.
  - El resto: Santander / Fiesta Rewards / Aeroméxico Platinum (43 / 22 / 35 %).

Nivel para comparar con Banxico: Free y LikeU = Clásica; Aeroméxico Blanca, Fiesta Rewards Oro y Santander Gold = Oro; las tres Platinum y World Elite = Platino. Esperado: 59.1 / 31.0 / 9.9 %.

### Activación de promociones (`activacion_promos_pct`)
A mayor ingreso, menor sensibilidad al precio: base por segmento y −5 pts cada vez que el ingreso se multiplica por *e* sobre la media del segmento. Según Banxico, el 55 % de los totaleros toma promociones a meses sin intereses contra el 40 % de los no-totaleros (Cuadro 2). Media ≈ 51 % como el original. Rango [0, 100].

`clamp(0, base(segmento) − 5·máx(0, ln(ingreso / media del segmento)) + (5 si totalero, −10 si no) + N(0, 18), 100)`, con bases Mass 52, Select 48, Premier 44.

### Fuentes
- Banco de México. (2026). *Indicadores básicos de tarjetas de crédito. Datos a diciembre de 2025.* Cuadros 1, 2, 4, 7, 8, 9, 10 y 11.
- Banco Santander México. (2026). [*Resumen ejecutivo del Informe Anual Integrado 2025*](https://www.santander.com.mx/personas/acerca-del-banco/responsabilidad-social/assets/pdf/informe-anual/resumen-ejecutivo-del-informe-anual-integrado-2025-v1.pdf).
- PopulationPyramid.net. (s.f.). *Pirámides de población del mundo desde 1950 a 2100: México 2026.*
- Dataset dummy original del reto (`dataset_dummy_tarjetas_santander_10000_clientes.csv.xlsx`), estadísticas de `EDA_segmentos.ipynb`.

### Supuestos propios
Los multiplicadores de ingreso por edad, la desviación de la satisfacción, los coeficientes de las fórmulas de NPS, totalero, e-commerce, logins y abandono, y las bases de promociones por segmento son supuestos del equipo. Se eligieron para respetar las cifras de arriba; no salen directamente de una fuente.

## Setup

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

N = 10_000
rng = np.random.default_rng(2026)
SALIDA = "data/dataset_regenerado_v2_10000_clientes.csv"

# Edad: pirámide México 2026 acotada a 18–75 (18-24 = 7/10 de 15-24; 75 = 1/16 de 75+)
RANGOS_EDAD = [(18, 24), (25, 34), (35, 44), (45, 54), (55, 64), (65, 74), (75, 75)]
PESOS_EDAD = np.array([15_571_098, 21_105_726, 18_729_814, 15_640_829, 12_087_127, 7_384_543, 270_410])
MULT_INGRESO = np.array([0.62, 0.88, 1.06, 1.15, 1.08, 0.86, 0.80])   # mismo orden que RANGOS_EDAD

PROMO_BASE = {"Mass": 52, "Select": 48, "Premier": 44}

COLUMNAS = ["customer_id", "edad", "segmento", "antiguedad_meses", "ingreso_mensual", "tarjeta_santander",
            "uso_tarjeta_pct", "transacciones_mes", "gasto_mensual", "ecommerce_pct", "pagos_puntuales_pct",
            "app_logins_mes", "nps", "satisfaccion", "abandono_riesgo", "activacion_promos_pct"]

logistica = lambda x: 1 / (1 + np.exp(-x))
z = lambda x: (x - x.mean()) / x.std()

## Diagrama de generación

Cada caja es una variable y cada flecha significa **"se calcula con"**. Las filas siguen el orden de generación: arriba las raíces, abajo lo que depende de más cosas. Pasa el cursor sobre una caja para ver su fórmula. `customer_id` no depende de nada (C###### en orden) y no aparece.

- **Azul:** raíces (se muestrean solas).
- **Gris:** columnas del CSV calculadas con fórmula.
- **Naranja:** salen del árbol de decisión.
- **Blanco, en cursiva:** variables internas que no se guardan.

In [2]:
TIPOS = {  # tipo: (fondo, borde, etiqueta de la leyenda)
    "raiz": ("#cde2fb", "#2a78d6", "Raíz"),
    "formula": ("#f0efec", "#52514e", "Calculada con fórmula"),
    "arbol": ("#fbe1d6", "#eb6834", "Árbol de decisión"),
    "interna": ("#fcfcfb", "#a3a29c", "Interna (no se guarda)"),
}
# nombre: (x, fila, tipo, descripción corta, fórmula para el hover). Posiciones elegidas para que ninguna flecha cruce una caja.
DIAGRAMA = {
    "edad": (3.25, 0, "raiz", "pirámide 2026", "Grupo por pirámide México 2026 (18–75); edad uniforme dentro del grupo"),
    "satisfaccion": (5.0, 0, "raiz", "N(4, 0.6)", "clip(N(4, 0.6), 1, 5), un decimal"),
    "antiguedad_meses": (3.25, 1, "formula", "U(1, tope por edad)", "uniforme(1, min(180, (edad − 18)·12 + 1))"),
    "ingreso_mensual": (1.75, 1, "formula", "log-normal × edad", "29,100 · multiplicador(edad) · exp(N(0, 0.47))"),
    "nps": (5.5, 1, "formula", "6.92 + 2.6·(sat − 4)", "clamp(0, round(6.92 + 2.6·(satisfacción − 4) + N(0, 1.4)), 10)"),
    "gasto_mensual": (4.25, 2, "formula", "≈ 39 % del ingreso", "min(0.39 · ingreso · exp(N(0, 0.25) − 0.25²/2), 1.1 · ingreso)"),
    "gasto / ingreso": (0.5, 3, "interna", "proporción", "gasto_mensual / ingreso_mensual"),
    "segmento": (2.25, 3, "arbol", "árbol 1", "Premier: gasto ≥ 22k y antig ≥ 24 m · Select: gasto ≥ 15.5k y antig ≥ 12 m · Mass: resto"),
    "transacciones_mes": (4.5, 3, "formula", "Poisson(gasto^0.3)", "máx(1, Poisson(18 · (gasto / 11,100)^0.3))"),
    "ecommerce_pct": (6.0, 3, "formula", "+ gasto, − edad", "clamp(0, 50 + 12·z(ln gasto) − 0.55·(edad − 41) + N(0, 20), 100)"),
    "totalero": (0.75, 4, "interna", "logística", "P = logística(0.80 − 4·(gasto/ingreso − 0.39) + 0.6·ln(ingreso/27,140) + 0.012·(edad − 40))"),
    "tarjeta_santander": (3.5, 4, "arbol", "árbol 2", "Según segmento, ingreso, gasto y antigüedad (ver árbol 2 abajo)"),
    "pagos_puntuales_pct": (3.0, 5, "formula", "100 − Exponencial", "Totalero: 100 − Exp(1.5) · No-totalero: 100 − Exp(11) − 12·máx(0, gasto/ingreso − 0.39) · recorte [70, 100]"),
    "uso_tarjeta_pct": (0.0, 5, "formula", "Beta(totalero, trans.)", "Beta(media = (0.24 totalero | 0.60 no) · (transacciones/18)^0.3, concentración 6) · 100"),
    "activacion_promos_pct": (1.75, 5, "formula", "base seg. ± totalero", "clamp(0, base(seg) − 5·máx(0, ln(ingreso/media seg)) + (5 | −10 si totalero) + N(0, 18), 100)"),
    "app_logins_mes": (5.75, 5, "formula", "Poisson(ecom., trans.)", "máx(1, Poisson(2.5 + 0.06·ecommerce + 0.24·transacciones))"),
    "abandono_riesgo": (5.0, 6, "formula", "logística(lealtad, …)", "logística(1.12 − 0.045·lealtad + 1.3·[uso < 10] + 0.06·(100 − pagos) + N(0, 0.6)); lealtad = 40·min(antig, 36)/36 + 30·nps/10 + 30·sat/5"),
}
FLECHAS = [  # (de, a): "a" se calcula con "de"
    ("edad", "antiguedad_meses"), ("edad", "ingreso_mensual"), ("edad", "totalero"), ("edad", "ecommerce_pct"),
    ("satisfaccion", "nps"), ("satisfaccion", "abandono_riesgo"),
    ("antiguedad_meses", "segmento"), ("antiguedad_meses", "tarjeta_santander"), ("antiguedad_meses", "abandono_riesgo"),
    ("ingreso_mensual", "gasto_mensual"), ("ingreso_mensual", "gasto / ingreso"), ("ingreso_mensual", "totalero"),
    ("ingreso_mensual", "tarjeta_santander"), ("ingreso_mensual", "activacion_promos_pct"),
    ("gasto_mensual", "gasto / ingreso"), ("gasto_mensual", "segmento"), ("gasto_mensual", "tarjeta_santander"),
    ("gasto_mensual", "transacciones_mes"), ("gasto_mensual", "ecommerce_pct"),
    ("gasto / ingreso", "totalero"), ("gasto / ingreso", "pagos_puntuales_pct"),
    ("nps", "abandono_riesgo"),
    ("segmento", "tarjeta_santander"), ("segmento", "activacion_promos_pct"),
    ("totalero", "pagos_puntuales_pct"), ("totalero", "uso_tarjeta_pct"), ("totalero", "activacion_promos_pct"),
    ("transacciones_mes", "app_logins_mes"), ("transacciones_mes", "uso_tarjeta_pct"),
    ("ecommerce_pct", "app_logins_mes"),
    ("pagos_puntuales_pct", "abandono_riesgo"), ("uso_tarjeta_pct", "abandono_riesgo"),
]
# Chequeo: el diagrama cubre todas las columnas del CSV (menos customer_id) y cada flecha une nodos existentes
assert set(DIAGRAMA) - {"gasto / ingreso", "totalero"} == set(COLUMNAS) - {"customer_id"}
assert all(a in DIAGRAMA and b in DIAGRAMA for a, b in FLECHAS)

MEDIO_ALTO = 0.17   # media altura de una caja, en unidades de fila
fig = go.Figure()
for a, b in FLECHAS:
    (xa, fa, *_), (xb, fb, *_) = DIAGRAMA[a], DIAGRAMA[b]
    fig.add_annotation(x=xb, y=-fb + MEDIO_ALTO, ax=xa, ay=-fa - MEDIO_ALTO, xref="x", yref="y", axref="x", ayref="y",
                       text="", showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.2, arrowcolor="#a3a29c")
for nombre, (x, fila, tipo, corto, formula) in DIAGRAMA.items():
    fondo, borde, _ = TIPOS[tipo]
    titulo = f"<i>{nombre}</i>" if tipo == "interna" else f"<b>{nombre}</b>"
    fig.add_annotation(x=x, y=-fila, text=f"{titulo}<br>{corto}", showarrow=False, width=140, align="center",
                       borderpad=5, font=dict(size=10, color="#0b0b0b"), bgcolor=fondo, bordercolor=borde, borderwidth=1.5,
                       hovertext=f"{nombre}: {formula}", hoverlabel=dict(bgcolor="white"))
for tipo, (fondo, borde, etiqueta) in TIPOS.items():   # leyenda
    fig.add_scatter(x=[None], y=[None], mode="markers", name=etiqueta,
                    marker=dict(symbol="square", size=14, color=fondo, line=dict(color=borde, width=1.5)))
fig.update_xaxes(visible=False, range=[-0.6, 6.6])
fig.update_yaxes(visible=False, range=[-6.5, 0.5])
fig.update_layout(title="Cómo se genera cada variable (flecha = se calcula con)", width=1300, height=900,
                  plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb", margin=dict(l=20, r=20, t=80, b=20),
                  legend=dict(orientation="h", y=1.05, x=1, xanchor="right"))
fig.show()

# Árbol de decisión basado en políticas

Árbol **determinístico** que traduce a reglas las políticas de `segmento` y `tarjeta_santander`. Son dos árboles encadenados: primero el segmento (gasto y antigüedad) y luego la tarjeta. Las hojas con varias tarjetas eligen al azar con las proporciones del original.

In [3]:
U = dict(
    premier_gasto=22_000, premier_antig=24,
    select_gasto=15_500, select_antig=12,
    mass_ingreso_alto=31_000,                    # el 25 % de mayor ingreso de Mass
    mass_gasto_alto=7_000,                       # LikeU vs Free
    select_gasto_media=None,                     # "media del segmento": se calcula al generar (abajo)
    select_antig_alta=72,
    we_ingreso=60_000, we_antig=60, we_gasto=30_000,
)
PLATINUMS = ("Santander Platinum", "Fiesta Rewards Platinum", "Aeromexico Platinum")
P_PLATINUMS = (0.43, 0.22, 0.35)                 # proporciones del original

def nodo(pregunta, *ramas):
    """ramas: (etiqueta, condición(cliente) -> bool, hijo). La primera condición que se cumple gana."""
    return {"pregunta": pregunta, "ramas": ramas}

def hoja(*tarjetas, p=None):
    return {"hoja": tarjetas, "p": p}

ARBOL_SEGMENTO = nodo(
    f"gasto ≥ {U['premier_gasto']:,}\ny antigüedad ≥ {U['premier_antig']} m",
    ("Sí", lambda c: c["gasto_mensual"] >= U["premier_gasto"] and c["antiguedad_meses"] >= U["premier_antig"], hoja("Premier")),
    ("No", lambda c: True, nodo(
        f"gasto ≥ {U['select_gasto']:,}\ny antigüedad ≥ {U['select_antig']} m",
        ("Sí", lambda c: c["gasto_mensual"] >= U["select_gasto"] and c["antiguedad_meses"] >= U["select_antig"], hoja("Select")),
        ("No", lambda c: True, hoja("Mass")),
    )),
)

ARBOL_TARJETA = nodo(
    "segmento",
    ("Mass", lambda c: c["segmento"] == "Mass", nodo(
        f"ingreso ≥ {U['mass_ingreso_alto']:,}",
        ("Sí", lambda c: c["ingreso_mensual"] >= U["mass_ingreso_alto"],
         hoja("Aeromexico Blanca", "Fiesta Rewards Oro", p=(0.63, 0.37))),
        ("No", lambda c: True, nodo(
            f"gasto ≥ {U['mass_gasto_alto']:,}",
            ("Sí", lambda c: c["gasto_mensual"] >= U["mass_gasto_alto"], hoja("LikeU")),
            ("No", lambda c: True, hoja("Free")),
        )),
    )),
    ("Select", lambda c: c["segmento"] == "Select", nodo(
        f"gasto ≥ media Select\ny antigüedad ≥ {U['select_antig_alta']} m",
        ("Sí", lambda c: c["gasto_mensual"] >= U["select_gasto_media"] and c["antiguedad_meses"] >= U["select_antig_alta"],
         hoja(*PLATINUMS, p=P_PLATINUMS)),
        ("No", lambda c: True, hoja("Santander Gold")),
    )),
    ("Premier", lambda c: c["segmento"] == "Premier", nodo(
        f"ingreso ≥ {U['we_ingreso']:,}\nantigüedad ≥ {U['we_antig']} m\ngasto ≥ {U['we_gasto']:,}",
        ("Sí", lambda c: c["ingreso_mensual"] >= U["we_ingreso"] and c["antiguedad_meses"] >= U["we_antig"]
                         and c["gasto_mensual"] >= U["we_gasto"], hoja("World Elite")),
        ("No", lambda c: True, hoja(*PLATINUMS, p=P_PLATINUMS)),
    )),
)

def recorrer(arbol, cliente, rng):
    """Devuelve (resultado, ruta). Si la hoja tiene varias tarjetas se elige al azar con sus proporciones."""
    ruta = []
    while "pregunta" in arbol:
        etiqueta, _, hijo = next(r for r in arbol["ramas"] if r[1](cliente))
        ruta.append(f"{arbol['pregunta'].replace(chr(10), ' ')} → {etiqueta}")
        arbol = hijo
    opciones = arbol["hoja"]
    return (opciones[0] if len(opciones) == 1 else str(rng.choice(opciones, p=arbol["p"]))), " | ".join(ruta)

def asignar(cliente, rng):
    cliente = dict(cliente)
    cliente["segmento"], _ = recorrer(ARBOL_SEGMENTO, cliente, rng)
    tarjeta, ruta = recorrer(ARBOL_TARJETA, cliente, rng)
    return cliente["segmento"], tarjeta, ruta

# Generación de 10,000 clientes

Orden: raíces (edad, satisfacción) → antigüedad, ingreso, gasto, NPS → segmento y tarjeta (árbol) → totalero (interna) → el resto. Semilla fija (`2026`), así que el resultado es reproducible.

In [4]:
# Raíces
idx = rng.choice(len(RANGOS_EDAD), size=N, p=PESOS_EDAD / PESOS_EDAD.sum())
lo, hi = np.array(RANGOS_EDAD).T
edad = rng.integers(lo[idx], hi[idx] + 1)
satisfaccion = np.clip(rng.normal(4, 0.6, N), 1, 5).round(1)

# Derivadas de las raíces
antiguedad = rng.integers(1, np.minimum(180, (edad - 18) * 12 + 1) + 1)
ingreso = (29_100 * MULT_INGRESO[idx] * np.exp(rng.normal(0, 0.47, N))).round(2)
gasto = np.minimum(0.39 * ingreso * np.exp(rng.normal(0, 0.25, N) - 0.25**2 / 2), 1.1 * ingreso).round(2)
nps = np.clip(np.round(6.92 + 2.6 * (satisfaccion - 4) + rng.normal(0, 1.4, N)), 0, 10).astype(int)

df = pd.DataFrame({"customer_id": [f"C{i:06d}" for i in range(1, N + 1)], "edad": edad,
                   "antiguedad_meses": antiguedad, "ingreso_mensual": ingreso, "gasto_mensual": gasto,
                   "nps": nps, "satisfaccion": satisfaccion})

# Segmento y tarjeta (árbol). La "media del segmento" de Select se calcula con los clientes ya segmentados.
registros = df.to_dict("records")
for c in registros:
    c["segmento"], _ = recorrer(ARBOL_SEGMENTO, c, rng)
df["segmento"] = [c["segmento"] for c in registros]
U["select_gasto_media"] = df.loc[df.segmento == "Select", "gasto_mensual"].mean()
df[["tarjeta_santander", "ruta"]] = [recorrer(ARBOL_TARJETA, c, rng) for c in registros]

# Totalero (interna, no se guarda)
ratio = gasto / ingreso
totalero = rng.random(N) < logistica(0.80 - 4 * (ratio - 0.39) + 0.6 * np.log(ingreso / 27_140) + 0.012 * (edad - 40))

# Comportamiento
df["transacciones_mes"] = np.maximum(1, rng.poisson(18 * (gasto / 11_100) ** 0.3))
df["ecommerce_pct"] = np.clip(50 + 12 * z(np.log(gasto)) - 0.55 * (edad - 41) + rng.normal(0, 20, N), 0, 100).round(1)
df["app_logins_mes"] = np.maximum(1, rng.poisson(2.5 + 0.06 * df.ecommerce_pct + 0.24 * df.transacciones_mes))

atraso = np.where(totalero, rng.exponential(1.5, N), rng.exponential(11, N) + 12 * np.maximum(0, ratio - 0.39))
df["pagos_puntuales_pct"] = np.clip(100 - atraso, 70, 100).round(1)

media_uso = np.where(totalero, 0.24, 0.60) * (df.transacciones_mes / 18) ** 0.3
df["uso_tarjeta_pct"] = np.clip((100 * rng.beta(6 * media_uso, 6 * (1 - media_uso))).round(2), 0.01, 99.99)  # (0, 100) estricto

lealtad = 40 * np.minimum(antiguedad, 36) / 36 + 30 * nps / 10 + 30 * satisfaccion / 5
df["abandono_riesgo"] = logistica(1.12 - 0.045 * lealtad + 1.3 * (df.uso_tarjeta_pct < 10)
                                  + 0.06 * (100 - df.pagos_puntuales_pct) + rng.normal(0, 0.6, N)).round(3)

media_seg = df.groupby("segmento").ingreso_mensual.transform("mean")
df["activacion_promos_pct"] = np.clip(df.segmento.map(PROMO_BASE) - 5 * np.maximum(0, np.log(ingreso / media_seg))
                                      + np.where(totalero, 5, -10) + rng.normal(0, 18, N), 0, 100).round(1)

clientes = df[COLUMNAS]
print(f"Media de gasto en Select: {U['select_gasto_media']:,.0f} MXN")

Media de gasto en Select: 18,696 MXN


## Validación

Reglas duras (deben cumplirse al 100 %) y **metas de la política**: cada cifra que la política dice que debe salir, con una tolerancia.

In [5]:
assert clientes.customer_id.is_unique and len(clientes) == N and not clientes.isna().any().any()
assert clientes.edad.between(18, 75).all()
assert (clientes.antiguedad_meses <= np.minimum(180, (clientes.edad - 18) * 12 + 1)).all()
assert (clientes.gasto_mensual <= 1.1 * clientes.ingreso_mensual + 0.01).all()
assert clientes.uso_tarjeta_pct.between(0, 100, inclusive="neither").all()
assert clientes.pagos_puntuales_pct.between(70, 100).all()
for col in ["ecommerce_pct", "activacion_promos_pct"]:
    assert clientes[col].between(0, 100).all(), col
assert clientes.abandono_riesgo.between(0, 1).all() and clientes.nps.between(0, 10).all()
assert clientes.satisfaccion.between(1, 5).all()

NIVEL = {"Free": "Clásica", "LikeU": "Clásica", "Aeromexico Blanca": "Oro", "Fiesta Rewards Oro": "Oro",
         "Santander Gold": "Oro", "World Elite": "Platino", **{t: "Platino" for t in PLATINUMS}}
seg = clientes.segmento.value_counts(normalize=True) * 100
nivel = clientes.tarjeta_santander.map(NIVEL).value_counts(normalize=True) * 100
c = clientes
METAS = [  # (meta, esperado según la política, obtenido, tolerancia)
    ("Mass %", 78.8, seg["Mass"], 2), ("Select %", 14.2, seg["Select"], 2), ("Premier %", 7.0, seg["Premier"], 1.5),
    ("Nivel Clásica % (Banxico 59.0)", 59.1, nivel["Clásica"], 2), ("Nivel Oro % (Banxico 31.1)", 31.0, nivel["Oro"], 2),
    ("Nivel Platino % (Banxico 9.8)", 9.9, nivel["Platino"], 1.5),
    ("Totaleros % (Banxico 69.2)", 68.4, totalero.mean() * 100, 2),
    ("Ingreso medio", 30_600, c.ingreso_mensual.mean(), 1_000),
    ("Ingreso p5", 11_700, c.ingreso_mensual.quantile(.05), 600), ("Ingreso p95", 61_500, c.ingreso_mensual.quantile(.95), 2_500),
    ("Gasto / ingreso medio", 0.39, (c.gasto_mensual / c.ingreso_mensual).mean(), 0.01),
    ("Ticket promedio (gasto total / transacciones totales)", 670, c.gasto_mensual.sum() / c.transacciones_mes.sum(), 60),
    ("NPS medio", 6.92, c.nps.mean(), 0.15),
    ("E-commerce medio %", 50, c.ecommerce_pct.mean(), 2),
    ("Logins medio", 10, c.app_logins_mes.mean(), 0.5),
    ("% con pagos puntuales < 80", 5.4, (c.pagos_puntuales_pct < 80).mean() * 100, 1),
    ("Uso de línea medio % (Banxico ≈ 36)", 36, c.uso_tarjeta_pct.mean(), 2),
    ("Abandono medio", 0.175, c.abandono_riesgo.mean(), 0.015),
    ("Activación de promos media %", 51, c.activacion_promos_pct.mean(), 2),
]
metas = pd.DataFrame(METAS, columns=["meta", "esperado", "obtenido", "tolerancia"]).set_index("meta")
metas["cumple"] = np.where((metas.obtenido - metas.esperado).abs() <= metas.tolerancia, "✓", "✗")
display(metas.round(3))
assert (metas.cumple == "✓").all(), "Alguna meta de la política no se cumple"

display(pd.crosstab(clientes.tarjeta_santander, clientes.segmento, margins=True, margins_name="Total"))
display(clientes.describe().T)
clientes.to_csv(SALIDA, index=False)
print("Guardado:", SALIDA)

,esperado,obtenido,tolerancia,cumple
meta,,,,
Mass %,78.800,78.800,2.000,✓
Select %,14.200,14.150,2.000,✓
Premier %,7.000,7.050,1.500,✓
Nivel Clásica % (Banxico 59.0),59.100,59.090,2.000,✓
Nivel Oro % (Banxico 31.1),31.000,30.970,2.000,✓
Nivel Platino % (Banxico 9.8),9.900,9.940,1.500,✓
Totaleros % (Banxico 69.2),68.400,68.660,2.000,✓
Ingreso medio,30600.000,30623.671,1000.000,✓
Ingreso p5,11700.000,11714.538,600.000,✓


segmento,Mass,Premier,Select,Total
tarjeta_santander,,,,
Aeromexico Blanca,1243,0,0,1243
Aeromexico Platinum,0,186,90,276
Fiesta Rewards Oro,728,0,0,728
Fiesta Rewards Platinum,0,125,69,194
Free,2520,0,0,2520
LikeU,3389,0,0,3389
Santander Gold,0,0,1126,1126
Santander Platinum,0,267,130,397
World Elite,0,127,0,127


,count,mean,std,min,25%,50%,75%,max
edad,10000.0,41.250200,15.433665,18.000,28.0000,40.000,53.0000,75.000
antiguedad_meses,10000.0,73.764000,52.966901,1.000,26.0000,65.000,117.0000,180.000
ingreso_mensual,10000.0,30623.671057,16047.696376,3231.990,19183.3825,27138.315,38346.0175,160264.180
uso_tarjeta_pct,10000.0,35.105648,23.750767,0.040,15.2200,30.460,52.3300,99.050
transacciones_mes,10000.0,17.817000,5.223145,3.000,14.0000,17.000,21.0000,44.000
gasto_mensual,10000.0,11911.121474,7080.098810,1007.270,6955.6000,10217.685,14965.7425,81959.970
ecommerce_pct,10000.0,49.652950,22.716181,0.000,33.9750,50.000,65.4000,100.000
pagos_puntuales_pct,10000.0,95.616080,6.624706,70.000,95.4000,98.300,99.4000,100.000
app_logins_mes,10000.0,9.731100,3.751746,1.000,7.0000,9.000,12.0000,32.000
nps,10000.0,6.849400,1.975325,0.000,6.0000,7.000,8.0000,10.000


Guardado: data/dataset_regenerado_v2_10000_clientes.csv


## Árboles y ejemplos

In [6]:
def dibujar_arbol(arbol, titulo):
    nodos, aristas, hojas_x = [], [], [0]

    def colocar(n, prof):
        if "hoja" in n:
            x = hojas_x[0]; hojas_x[0] += 1
            etiquetas = n["hoja"] if n["p"] is None else [f"{t} ({p:.0%})" for t, p in zip(n["hoja"], n["p"])]
            nodos.append((x, -prof, "<br>".join(etiquetas), True))
            return x
        xs = [colocar(h, prof + 1) for _, _, h in n["ramas"]]
        x = sum(xs) / len(xs)
        nodos.append((x, -prof, n["pregunta"].replace("\n", "<br>"), False))
        for (etq, _, _), xh in zip(n["ramas"], xs):
            aristas.append((x, -prof, xh, -prof - 1, etq))
        return x

    colocar(arbol, 0)
    fig = go.Figure()
    for x0, y0, x1, y1, etq in aristas:
        fig.add_shape(type="line", x0=x0, y0=y0, x1=x1, y1=y1, line=dict(color="#a3a29c", width=1.5), layer="below")
        fig.add_annotation(x=(x0 + x1) / 2, y=(y0 + y1) / 2, text=f"<b>{etq}</b>", showarrow=False,
                           font=dict(size=11, color="#52514e"), bgcolor="#fcfcfb")
    for x, y, texto, es_hoja in nodos:
        fig.add_annotation(x=x, y=y, text=texto, showarrow=False, align="center", borderpad=6,
                           font=dict(size=11, color="#0b0b0b"),
                           bgcolor="#cde2fb" if es_hoja else "#f0efec",
                           bordercolor="#2a78d6" if es_hoja else "#52514e", borderwidth=1.5)
    profundidad = -min(y for _, y, _, _ in nodos)
    fig.update_xaxes(visible=False, range=[-0.7, hojas_x[0] - 0.3])
    fig.update_yaxes(visible=False, range=[-profundidad - 0.5, 0.5])
    fig.update_layout(title=titulo, height=170 * (profundidad + 1), plot_bgcolor="#fcfcfb",
                      paper_bgcolor="#fcfcfb", margin=dict(l=20, r=20, t=60, b=20), showlegend=False)
    return fig

dibujar_arbol(ARBOL_SEGMENTO, "Árbol 1 — Asignación de segmento (gasto y antigüedad)").show()
dibujar_arbol(ARBOL_TARJETA, f"Árbol 2 — Asignación de tarjeta (media Select = {U['select_gasto_media']:,.0f} MXN)").show()

In [7]:
# Un cliente por hoja del árbol; los assert fallan si un cambio de umbral rompe la política.
m = U["select_gasto_media"]
ejemplos = pd.DataFrame([
    dict(customer_id="C000001", ingreso_mensual=15_000, gasto_mensual=5_000, antiguedad_meses=6),     # Mass → Free
    dict(customer_id="C000002", ingreso_mensual=20_000, gasto_mensual=9_000, antiguedad_meses=8),     # Mass → LikeU
    dict(customer_id="C000003", ingreso_mensual=40_000, gasto_mensual=12_000, antiguedad_meses=40),   # Mass → Blanca/Oro
    dict(customer_id="C000004", ingreso_mensual=40_000, gasto_mensual=16_000, antiguedad_meses=100),  # Select, gasto < media → Gold
    dict(customer_id="C000005", ingreso_mensual=50_000, gasto_mensual=m + 500, antiguedad_meses=100), # Select → Platinum
    dict(customer_id="C000006", ingreso_mensual=50_000, gasto_mensual=m + 500, antiguedad_meses=30),  # Select, poca antigüedad → Gold
    dict(customer_id="C000007", ingreso_mensual=50_000, gasto_mensual=25_000, antiguedad_meses=50),   # Premier → Platinum
    dict(customer_id="C000008", ingreso_mensual=90_000, gasto_mensual=40_000, antiguedad_meses=150),  # Premier → World Elite
])
rng_ej = np.random.default_rng(42)
ejemplos[["segmento", "tarjeta_santander", "ruta"]] = [asignar(c, rng_ej) for c in ejemplos.to_dict("records")]

esperado = ["Free", "LikeU", {"Aeromexico Blanca", "Fiesta Rewards Oro"}, "Santander Gold",
            set(PLATINUMS), "Santander Gold", set(PLATINUMS), "World Elite"]
for t, e in zip(ejemplos.tarjeta_santander, esperado):
    assert t in e if isinstance(e, set) else t == e, (t, e)
assert list(ejemplos.segmento) == ["Mass"] * 3 + ["Select"] * 3 + ["Premier"] * 2

pd.set_option("display.max_colwidth", None)
display(ejemplos)

,customer_id,ingreso_mensual,gasto_mensual,antiguedad_meses,segmento,tarjeta_santander,ruta
0,C000001,15000,5000.000000,6,Mass,Free,"segmento → Mass | ingreso ≥ 31,000 → No | gasto ≥ 7,000 → No"
1,C000002,20000,9000.000000,8,Mass,LikeU,"segmento → Mass | ingreso ≥ 31,000 → No | gasto ≥ 7,000 → Sí"
2,C000003,40000,12000.000000,40,Mass,Fiesta Rewards Oro,"segmento → Mass | ingreso ≥ 31,000 → Sí"
3,C000004,40000,16000.000000,100,Select,Santander Gold,segmento → Select | gasto ≥ media Select y antigüedad ≥ 72 m → No
4,C000005,50000,19195.633739,100,Select,Fiesta Rewards Platinum,segmento → Select | gasto ≥ media Select y antigüedad ≥ 72 m → Sí
5,C000006,50000,19195.633739,30,Select,Santander Gold,segmento → Select | gasto ≥ media Select y antigüedad ≥ 72 m → No
6,C000007,50000,25000.000000,50,Premier,Aeromexico Platinum,"segmento → Premier | ingreso ≥ 60,000 antigüedad ≥ 60 m gasto ≥ 30,000 → No"
7,C000008,90000,40000.000000,150,Premier,World Elite,"segmento → Premier | ingreso ≥ 60,000 antigüedad ≥ 60 m gasto ≥ 30,000 → Sí"


## Notas de implementación (IA)

> 🤖 Escritas por una **IA (Claude, de Anthropic)**. Puntos en los que la política admitía más de una lectura:

- **Satisfacción:** la normal se recorta a [1, 5] (`clip`), así que hay un pequeño pico en 5.0.
- **`Exponencial(1.5)` y `Exponencial(11)`:** el número es la **media** del atraso (en puntos porcentuales). Con esta lectura sale la meta de ≈ 5.4 % de clientes con menos de 80 % de pagos a tiempo.
- **"Media del segmento" en Select:** se calcula con los clientes ya segmentados. Por eso primero se asigna el segmento a todos y luego la tarjeta.
- **Uso de tarjeta:** se recorta a [0.01, 99.99] después de redondear, para garantizar el intervalo abierto (0, 100).
- **`totalero`** se usa para generar pagos, uso y promociones, pero no se guarda en el CSV, como pide la política.
- La celda de **metas** compara cada cifra de la política con lo que sale y falla si alguna se sale de su tolerancia. Si cambian un coeficiente, esa celda les avisa.